# Week 5 · Build, test, and train a small CNN

**Challenge:** can an inspection camera recognise an L, T, or + when it moves?

Implement convolution and pooling in NumPy, check them against PyTorch, train a tiny CNN, implement a residual block, and investigate movement and augmentation. Start with worksheet activity 1. Work in pairs; switch driver after activity 2.

Data generation, training, and plotting are supplied. Complete the marked `TODO` gaps; unfinished functions return `None` and their checks explain what is missing. There are no hidden model solutions or automatic fallbacks. Run cells from top to bottom after edits. The technical repository checker is not a student answer checker.

Use NumPy, Matplotlib, and CPU PyTorch only. No downloads, torchvision, GPU, or book package are needed.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from dlexercises.week05 import (
    make_data, fit, show_images, plot_losses,
    parameter_count, report,
)

torch.set_num_threads(2)
torch.manual_seed(7)
print("Ready: NumPy", np.__version__, "| PyTorch", torch.__version__, "| CPU")

## 1. Build and test a convolution

Implement **valid cross-correlation** for a single 2D image and one 2D kernel: stride 1, no padding, scalar bias. This is the operation libraries usually call convolution; do **not** flip the kernel.

For input `(H, W)` and kernel `(Kh, Kw)`, return `(H-Kh+1, W-Kw+1)`. Use two loops over output positions and multiply each image patch elementwise by the kernel, sum, and add the bias. Do not call a convolution library here.

**Before coding:** calculate the top-left output for the example below. Why does the bias get added once per output rather than once per input pixel?

In [ ]:
def conv2d_numpy(image, kernel, bias=0.0):
    h, w = image.shape
    kh, kw = kernel.shape
    output = np.zeros((h - kh + 1, w - kw + 1))

    for row in range(output.shape[0]):
        for col in range(output.shape[1]):
            patch = image[row:row + kh, col:col + kw]
            # TODO: replace the next line with:
            # output[row, col] = the sum of patch * kernel, plus bias
            return None
    return output

image = np.array([[1, 1, 0], [1, 0, 0], [1, 1, 1]], dtype=float)
kernel = np.array([[1, -1], [1, -1]], dtype=float)
result = conv2d_numpy(image, kernel)
if result is not None:
    np.testing.assert_allclose(result, [[1, 1], [1, 0]])
    print("Convolution check passed:", result)
else:
    print("Complete the marked line, then rerun this cell.")

### The same operation in PyTorch

Use the same image, kernel, and bias. PyTorch adds two dimensions: batch and channel. Here both have size 1. The weights have shape `(out_channels, in_channels, height, width)`.

**Explain:** why can this filter accept a larger image without learning more weights?

In [ ]:
layer = nn.Conv2d(1, 1, kernel_size=2).double()
with torch.no_grad():
    layer.weight.copy_(torch.tensor(kernel).reshape(1, 1, 2, 2))
    layer.bias.fill_(0.5)

x = torch.tensor(image).reshape(1, 1, 3, 3)
pytorch_result = layer(x).detach().numpy()[0, 0]
numpy_result = conv2d_numpy(image, kernel, bias=0.5)
if numpy_result is not None:
    np.testing.assert_allclose(numpy_result, pytorch_result)
    print("NumPy and PyTorch agree, including the bias.")

**Your explanation:** What kind of edge does this kernel detect? What happens to the response if the edge moves? Which values would a trained CNN learn?

_Response:_ …

## 2. Build pooling

Complete the marked line to take the maximum in each 2×2 window. The loops move two pixels at a time. Ignore any leftover row or column.

**Predict:** what information does pooling discard? Could this hurt when recognising a very small mark?

In [ ]:
def maxpool2d_numpy(image):
    h, w = image.shape
    output = np.zeros((h // 2, w // 2))

    for row in range(output.shape[0]):
        for col in range(output.shape[1]):
            patch = image[2*row:2*row + 2, 2*col:2*col + 2]
            # TODO: replace the next line with:
            # output[row, col] = the maximum value in patch
            return None
    return output

pool_input = np.array([[1, 4, 2, 3], [0, 2, 8, 1],
                       [5, 1, 3, 0], [2, 6, 4, 9]])
result = maxpool2d_numpy(pool_input)
if result is not None:
    np.testing.assert_allclose(result, [[4, 8], [6, 9]])
    print("Pooling check passed:", result)
else:
    print("Complete the marked line, then rerun this cell.")

**Your explanation:** Does max pooling have learned weights? During backpropagation, which input in a window receives the output gradient (assuming a unique maximum)?

_Response:_ …

## 3. Build the library CNN

The supplied generator creates balanced grayscale **L, T, +** images. Thickness, size, brightness, and noise vary independently of the label. All symbols share a fixed orientation. Centered and shifted development sets contain the same examples with different positions. Train and development use different seeds.

These are controlled synthetic images, not evidence of real camera performance. Inspect examples before training.

In [ ]:
train = make_data(120, seed=11)
valid = make_data(40, seed=22)
shifted_valid = make_data(40, seed=22, shifted=True)
show_images(valid)
show_images(shifted_valid)

Complete the two convolution definitions below. The forward pass is supplied, one step per line. The output is **three raw logits**; `CrossEntropyLoss` already handles the log-softmax calculation.

| Operation | Output excluding batch |
|---|---|
| Input | `1 × 32 × 32` |
| Conv 3×3, 1→8, padding 1; ReLU | `8 × 32 × 32` |
| MaxPool 2×2, stride 2 | `8 × 16 × 16` |
| Conv 3×3, 8→16, padding 1; ReLU | `16 × 16 × 16` |
| MaxPool 2×2, stride 2 | `16 × 8 × 8` |
| Average over height and width | `16` |
| Linear 16→3 | `3` |

**Predict:** count parameters of the second convolution, including bias. Calculate the receptive field just after the second max pool. How does global average pooling change the classifier parameter count compared with flattening `16 × 8 × 8`?

In [ ]:
class TinyCNN(nn.Module):
    def __init__(self):
        super().__init__()
        # TODO: 3x3 convolution, 1 input channel, 8 output channels, padding=1
        self.conv1 = None
        # TODO: 3x3 convolution, 8 input channels, 16 output channels, padding=1
        self.conv2 = None
        self.pool = nn.MaxPool2d(2)
        self.classifier = nn.Linear(16, 3)

    def forward(self, x):
        if self.conv1 is None or self.conv2 is None:
            return None  # Allows the unfinished starter to run.
        x = torch.relu(self.conv1(x))
        x = self.pool(x)
        x = torch.relu(self.conv2(x))
        x = self.pool(x)
        x = x.mean(dim=(2, 3))  # Global average pooling: one value per channel.
        return self.classifier(x)

cnn = TinyCNN()
cnn_ready = cnn(torch.zeros(4, 1, 32, 32)) is not None
if cnn_ready:
    assert cnn(torch.zeros(4, 1, 32, 32)).shape == (4, 3)
    print("Output shape is correct. Parameters:", parameter_count(cnn))
else:
    print("Define conv1 and conv2, then rerun.")

**Your prediction and reasoning:** …

## 4. Train and inspect learning

The supplied training function uses minibatches of 60, Adam with learning rate 0.003, and 18 epochs. It restores the checkpoint with lowest **centered-validation cross-entropy**. Train and validation losses are measured on clean images in evaluation mode.

Read the training loop in `src/dlexercises/week05.py`: locate `zero_grad()`, the forward pass, the loss, `backward()`, and `step()`. Explain their roles to your partner. A shared convolution weight receives gradient contributions from all positions where it was used.

**Before running:** what should happen to training loss? Must validation loss fall at every epoch?

In [ ]:
runs = {}
if cnn_ready:
    runs["CNN"] = fit(TinyCNN, train, valid)
    plot_losses(runs)
    print("First filter tensor changed by L2 norm:", runs["CNN"]["weight_change"])
    print("Selected epoch:", runs["CNN"]["best_epoch"])
    show_images(valid, runs["CNN"]["model"], mistakes_only=True)
else:
    print("Finish the CNN before training.")

**Record:** How did the losses change? Did the first filter tensor change? If training loss falls while validation loss rises, give one plausible explanation and one next check.

_Response:_ …

## 5. Implement a residual block

Start with the simplest case: input and output have the **same shape**. Two 3×3 convolutions preserve the channel count and image size. Complete the addition:

```
residual = Conv → ReLU → Conv
output = ReLU(residual + input)
```

The convolution definitions are supplied so you can focus on the skip connection. This is a simplified block without BatchNorm, not the complete ResNet-18 block.

**Predict:** if the residual is zero, does this block return `x` or `ReLU(x)`?

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, 3, padding=1)
        self.conv2 = nn.Conv2d(channels, channels, 3, padding=1)

    def forward(self, x):
        residual = torch.relu(self.conv1(x))
        residual = self.conv2(residual)
        # TODO: replace None with the sum of residual and x.
        combined = None
        if combined is None:
            return None
        return torch.relu(combined)

block = ResidualBlock(8)
x = torch.randn(2, 8, 16, 16)
result = block(x)
residual_ready = result is not None
if residual_ready:
    assert result.shape == x.shape
    print("Residual block preserves the input shape.")
else:
    print("Complete the addition, then rerun.")

**Explain:** Why can we add these two tensors? What happens if the learned branch produces zero? How does the shortcut provide a direct path for gradients?

_Response:_ …

**Follow-up:** if the branch halves the image size and doubles the channels, the original input can no longer be added directly. What stride and channel counts would a 1×1 shortcut convolution need?

### Optional: train with your residual block

The wrapper below puts your block after a stem convolution. This is a small residual CNN. We omit BatchNorm here to focus on the addition; standard ResNet basic blocks also use normalization. The models differ in depth and parameter count, so a score difference alone does not isolate the effect of shortcuts.

In [ ]:
class ResidualCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.stem = nn.Conv2d(1, 8, 3, padding=1)
        self.block = ResidualBlock(8)
        self.pool = nn.MaxPool2d(2)
        self.classifier = nn.Linear(8, 3)

    def forward(self, x):
        x = self.pool(torch.relu(self.stem(x)))
        x = self.block(x)
        x = x.mean(dim=(2, 3))
        return self.classifier(x)

RUN_RESIDUAL_TRAINING = False  # Optional: enable after completing the block.
if RUN_RESIDUAL_TRAINING and residual_ready:
    runs["Residual CNN"] = fit(ResidualCNN, train, valid)
    report(runs, valid, shifted_valid)

## 6. Movement, augmentation, and limitations

**Predict first:** will the CNN give the same prediction when a symbol moves eight pixels? Convolution is approximately translation equivariant away from boundaries at stride 1. Pooling, downsampling, padding, and the finite image boundary complicate this property. Global average pooling encourages robustness but does not guarantee identical predictions.

Measure centered and shifted development accuracy. Then train the **same TinyCNN** with random translations of at most eight pixels in each direction. Symbols remain fully visible. Only training images are augmented. Paired runs use identical initialization, minibatch order, epochs, and optimizer. Checkpoint selection remains centered-validation loss.

If the original model already handles the shifts well, that is a valid result: explain what might support its robustness. Do not invent a failure or assume augmentation must help.

In [ ]:
if cnn_ready:
    runs["CNN + translations"] = fit(TinyCNN, train, valid, augment=True)
    report(runs, valid, shifted_valid)
    plot_losses(runs)
    show_images(shifted_valid, runs["CNN"]["model"], mistakes_only=True)
else:
    print("Complete and train TinyCNN first.")

**Your results and explanation**

- Centered and shifted validation accuracy before augmentation: …
- Centered and shifted validation accuracy after augmentation: …
- Did the result match your prediction? Explain using the architecture or data: …
- Which factors are controlled? What would another random seed tell you? …
- Is a horizontal flip or rotation label-preserving for this inspection task? State the intended label definition: …

### Freeze a recommendation, then evaluate once

Choose between the two TinyCNN runs using validation evidence. Write a recommendation **before enabling** the test cell. The held-out test set is generated only here, with a fresh seed. Use its results for final reporting, not further selection. This is a classroom convention, not an access-control mechanism.

**My chosen configuration, selection criterion, and reason:** …

In [ ]:
CHOSEN_RUN = "CNN + translations"  # Choose using validation evidence.
RUN_FINAL_TEST = False  # Enable after writing your recommendation above.
if RUN_FINAL_TEST:
    test_centered = make_data(80, seed=9301)
    test_shifted = make_data(80, seed=9301, shifted=True)
    report({CHOSEN_RUN: runs[CHOSEN_RUN]}, test_centered, test_shifted)

## Individual reflection — return to the worksheet

Answer its final diagnosis and transfer questions. Distinguish what your experiments measured from what you expect in real images. Confident predictions and high accuracy on synthetic symbols do not establish robustness to new fonts, rotations, backgrounds, or camera conditions.

### Optional investigation: a misleading corner mark

Propose a training set in which corner-mark brightness predicts the class. Design paired development images that keep the symbol identical but change the mark. What result would support dependence on the mark? Propose one targeted training intervention and a controlled comparison. This is a design extension; no additional implementation is required for the core lab.

### Conventions and attribution

Original synthetic symbols generated for this course; no external image dataset. This notebook accompanies the supplied Lesson 5: CNNs, ResNets, and Augmentation. Shapes use NCHW; convolution means cross-correlation; the residual task uses a simplified post-activation block without BatchNorm. NumPy implementations are forward-only; PyTorch autograd computes training gradients. Source and prose reuse follow the repository LICENSE.md.